# NYC Congestion Pricing — SubPop Full Pipeline (Colab)

**Runtime required:** GPU (L4 recommended; T4 works for zero-shot; A100 for fine-tuning)

In Colab: Runtime → Change runtime type → L4 GPU

---
**Cell order (run top to bottom):**
1. Mount Drive + define helpers + restore outputs
2. Config — set active models, formats, and stages *(run every new session)*
3. Install dependencies
4. HuggingFace login
5. (3b) Download SubPop pre-training data (Pew-style)
6. (3c) Per-model QA pre-training (needed for sequential fine-tuning)
7. (4) Zero-shot baselines — loops over ACTIVE_MODELS × ACTIVE_FORMATS
8. (4b) Statistical bounds (shared, model-agnostic)
9. (5) Prepare CMS fine-tuning data (shared, model-agnostic)
10. (6) Base CMS fine-tuning + inference — loops over ACTIVE_MODELS × ACTIVE_FORMATS
11. (7) Sequential fine-tuning + inference — loops over ACTIVE_MODELS × ACTIVE_FORMATS
12. (8) Evaluation per model

**Output layout:**
```
approach2_outputs/cms/              <- shared permanent dir (git-tracked, matches cms_dataset config)
    cms_survey_distributions.csv    <- step1 ground truth
    cms_demographics.csv            <- step1 demographics
    cms_steering_prompts.json       <- step1 steering prompts
    cms_question_split.json         <- step1 train/val/test split
    cms_questions.json              <- step1 question definitions
    cms_subgroup_weights.csv        <- step1 subgroup weights
    statistical_bounds.csv          <- step2 (computed once)
    finetuning_data/                <- step3 (computed once, deterministic)
        cms_QA_train.csv  ...

approach2_outputs_YYYYMMDD/cms/     <- run-specific outputs (Drive-backed per session)
    llama/                          <- Llama-specific outputs
        distributions_QA.csv  ...   <- step2 zero-shot
        checkpoints_QA/             <- Cell 6 LoRA checkpoint
        results_finetuned_QA.csv    <- Cell 6 inference output
        evaluation/                 <- step6 tables, coverage manifest, and heatmaps
    qwen/                           <- Qwen-specific outputs
        ...
```

## Cell 1 — Mount Drive, define helpers, restore per-model outputs

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, shutil, subprocess, datetime
from zoneinfo import ZoneInfo

# ── MODELS REGISTRY ────────────────────────────────────────────────────────
# All supported models. Cell 1 uses this to scaffold Drive dirs and restore
# per-model outputs. Config cell (below) selects which subset to actually run.
MODELS = {
    'llama': 'meta-llama/Llama-3.1-8B',
    'qwen':  'Qwen/Qwen2.5-7B',
}

REPO_DIR        = '/content/Subpop_replication'
DRIVE_BASE      = '/content/drive/MyDrive/Congestion Pricing Project/Subpop_replication'
OUTPUT_DATE_TAG = globals().get('OUTPUT_DATE_TAG', datetime.datetime.now(ZoneInfo('America/New_York')).strftime('%Y%m%d'))
OUTPUT_ROOT     = f'approach2_outputs_{OUTPUT_DATE_TAG}'

# ── SHARED PERMANENT DATA DIRECTORY ─────────────────────────────────────────
# Ground truth CSVs, steering prompts, demographics, finetuning_data/ all live
# here. This path is committed to git and must match what cms_dataset reads:
#   subpop-main/subpop/train/configs/datasets.py
#   -> train_split: "../approach2_outputs/cms/finetuning_data/cms_{steering_type}_train.csv"
CMS_SHARED_DIR = f'{REPO_DIR}/approach2_outputs/cms'
CMS_SHARED_REL = 'approach2_outputs/cms'   # relative from REPO_DIR, for subprocess args


# ── PATH HELPERS (model-specific run outputs) ────────────────────────────────
# model_tag=None  -> shared permanent dir (should not be used for run outputs)
# model_tag='llama' -> OUTPUT_ROOT/cms/{model_tag}/ for this run's model outputs

def cms_local(rel_path='', model_tag=None):
    '''Absolute local path for model-specific run outputs.'''
    base = f'{REPO_DIR}/{OUTPUT_ROOT}/cms'
    if model_tag:
        base = f'{base}/{model_tag}'
    return base if not rel_path else f'{base}/{rel_path}'


def cms_rel(rel_path='', model_tag=None):
    '''Relative path (from REPO_DIR) for model-specific run outputs.'''
    base = f'{OUTPUT_ROOT}/cms'
    if model_tag:
        base = f'{base}/{model_tag}'
    return base if not rel_path else f'{base}/{rel_path}'


def cms_drive_rel(rel_path='', model_tag=None):
    '''Drive-relative path for save/restore of model-specific run outputs.'''
    base = f'{OUTPUT_ROOT}/cms'
    if model_tag:
        base = f'{base}/{model_tag}'
    return base if not rel_path else f'{base}/{rel_path}'


# ── I/O HELPERS ─────────────────────────────────────────────────────────────

def save_to_drive(local, drive_rel, label=''):
    dst = f'{DRIVE_BASE}/{drive_rel}'
    if os.path.isdir(local):
        os.makedirs(dst, exist_ok=True)
        shutil.copytree(local, dst, dirs_exist_ok=True)
    else:
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copy2(local, dst)
    print(f'  checkmark {label or drive_rel} saved to Drive')


def restore_from_drive(drive_rel, local, label=''):
    src = f'{DRIVE_BASE}/{drive_rel}'
    if not os.path.exists(src):
        return False
    if os.path.isdir(src) and not os.listdir(src):
        return False
    if os.path.isdir(src):
        os.makedirs(local, exist_ok=True)
        shutil.copytree(src, local, dirs_exist_ok=True)
    else:
        os.makedirs(os.path.dirname(local), exist_ok=True)
        shutil.copy2(src, local)
    print(f'  checkmark {label or drive_rel} restored from Drive')
    return True


def save_cms(rel_path, label='', model_tag=None):
    '''Save a model-specific run artifact to Drive.'''
    save_to_drive(
        cms_local(rel_path, model_tag),
        cms_drive_rel(rel_path, model_tag),
        label or rel_path,
    )


def exists_nonempty(path):
    if not os.path.exists(path):
        return False
    if os.path.isdir(path):
        return any(True for _ in os.scandir(path))
    return os.path.getsize(path) > 0


def should_run(stage, outputs, force=False):
    paths = outputs if isinstance(outputs, list) else [outputs]
    if force:
        print(f'[FORCE] {stage}: force flag set — running.')
        return True
    missing = [p for p in paths if not exists_nonempty(p)]
    if missing:
        names = [os.path.basename(m.rstrip('/')) for m in missing]
        print(f'[RUN]   {stage}: output(s) missing -> {names}')
        return True
    print(f'[SKIP]  {stage}: all outputs present — skipping.')
    return False


def require_exists(path, produced_by=''):
    if not path or not exists_nonempty(path):
        msg = f'Required path not found: {path}'
        if produced_by:
            msg += f'  |  Produced by: {produced_by}'
        raise FileNotFoundError(msg)
    return path


def latest_subdir(parent_glob):
    import glob
    dirs = sorted(glob.glob(parent_glob))
    return dirs[-1] if dirs else None


def make_cuda_env():
    from google.colab import userdata as _ud
    env = os.environ.copy()
    hf_tok = _ud.get('HF_TOKEN')
    if hf_tok:
        env['HF_TOKEN'] = hf_tok
        env['HUGGING_FACE_HUB_TOKEN'] = hf_tok
    else:
        print('  WARNING: HF_TOKEN Colab secret not set; public models may still work, gated models will fail.')
    env['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
    return env


def run_cmd(cmd, cwd=None):
    result = subprocess.run(cmd, cwd=cwd, capture_output=False, text=True)
    if result.returncode != 0:
        raise RuntimeError(f'Command failed: {cmd}')


# ── DRIVE DIRECTORY SCAFFOLDING (model-specific outputs only) ─────────────────
# Shared source artifacts come from git. finetuning_data/ is generated by Cell 5
# under the shared CMS directory; no Drive scaffolding is needed for shared files.

os.makedirs(f'{DRIVE_BASE}/pretrain_checkpoints', exist_ok=True)
for _mtag in MODELS:
    for _sub in [
        '',
        'checkpoints_QA', 'checkpoints_BIO', 'checkpoints_PORTRAY',
        'checkpoints_QA_seq', 'checkpoints_BIO_seq', 'checkpoints_PORTRAY_seq',
        'evaluation',
    ]:
        os.makedirs(f'{DRIVE_BASE}/{cms_drive_rel(_sub, _mtag)}', exist_ok=True)
    os.makedirs(f'{DRIVE_BASE}/pretrain_checkpoints/{_mtag}/checkpoints_QA', exist_ok=True)


# ── REPO CLONE / PULL ────────────────────────────────────────────────────────

if not os.path.exists(REPO_DIR):
    run_cmd(['git', 'clone', 'https://github.com/Twyla123/Subpop_replication.git', REPO_DIR])
else:
    run_cmd(['git', 'pull'], cwd=REPO_DIR)


# ── RESTORE DRIVE: model-specific run outputs only ───────────────────────────
# Shared source artifacts are restored from git, not Drive.
# Cell 5 regenerates finetuning_data/ under the shared CMS directory.
# Per-model outputs (distributions, checkpoints, results) are restored from Drive.

print(f'\nOutput root: {OUTPUT_ROOT}')
print('Checking Drive for per-model outputs...')
for _mtag in MODELS:
    restore_from_drive(cms_drive_rel('', _mtag), cms_local('', _mtag), f'{_mtag} outputs')
    restore_from_drive(
        f'pretrain_checkpoints/{_mtag}/checkpoints_QA',
        f'{REPO_DIR}/subpop-main/data/subpop-pretrain/{_mtag}/checkpoints_QA',
        f'{_mtag} pretrain QA checkpoint',
    )

# Verify shared data exists (comes from git clone)
for _f in ['cms_survey_distributions.csv', 'cms_demographics.csv',
           'cms_steering_prompts.json', 'cms_question_split.json',
           'cms_questions.json', 'cms_subgroup_weights.csv']:
    _p = f'{CMS_SHARED_DIR}/{_f}'
    if os.path.exists(_p):
        print(f'  ok  {_f}')
    else:
        raise FileNotFoundError(f'Missing required shared CMS artifact: {_p}')

print('Cell 1 complete.')

## Config Cell — Edit here to control what runs

Run this cell **after Cell 1** at the start of every session.  
All other cells read from `ACTIVE_MODELS`, `ACTIVE_FORMATS`, `RUN_*`, and `FORCE`.

**Directory contract:**
- Shared input data → `CMS_SHARED_DIR` = `approach2_outputs/cms/` (permanent, git-tracked)
- Per-model run outputs → `cms_local(rel, model_tag)` = `OUTPUT_ROOT/cms/{model_tag}/` (Drive-backed)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# PIPELINE CONFIG — edit this cell to control what runs.
# ═══════════════════════════════════════════════════════════════════════════

# ── Which models to run ────────────────────────────────────────────────────
# Remove a tag to skip that model entirely.
# Full HF model names are in MODELS (Cell 1).
ACTIVE_MODELS = ['llama', 'qwen']     # e.g. ['llama'] or ['qwen'] or both

# ── Which prompt formats to run ────────────────────────────────────────────
# Applies to zero-shot, base fine-tuning, and sequential fine-tuning.
ACTIVE_FORMATS = ['QA', 'BIO', 'PORTRAY']   # any subset

# ── Which pipeline stages to run ───────────────────────────────────────────
RUN_PRETRAIN_DOWNLOAD = True   # Cell 3b: download filtered Pew SubPop data
RUN_PRETRAIN_TRAIN    = True   # Cell 3c: QA pre-training per model (for sequential)
RUN_ZEROSHOT          = True   # Cell 4:  zero-shot baselines
RUN_FINETUNE          = True   # Cell 6:  base CMS fine-tuning
RUN_SEQUENTIAL        = True   # Cell 7:  sequential (pretrain -> CMS) fine-tuning

# ── FORCE flags: set True to re-run even if outputs already exist ──────────
FORCE = {
    'pretrain_download': True,
    'pretrain_train':    True,
    'zeroshot':          True,
    'bounds':            True,
    'step3':             True,
    'finetune':          True,
    'sequential':        True,
    'eval':              True,
}

# ── Sync OUTPUT_ROOT in case session was resumed mid-run ───────────────────
import datetime as _dt
OUTPUT_DATE_TAG = globals().get('OUTPUT_DATE_TAG', _dt.date.today().strftime('%Y%m%d'))
OUTPUT_ROOT     = f'approach2_outputs_{OUTPUT_DATE_TAG}'

# ── Summary ────────────────────────────────────────────────────────────────
print('Pipeline config loaded.')
print(f'  Active models  : {ACTIVE_MODELS}')
for _m in ACTIVE_MODELS:
    print(f'    {_m:8s} -> {MODELS[_m]}')
print(f'  Active formats : {ACTIVE_FORMATS}')
print(f'  Stages         : zeroshot={RUN_ZEROSHOT}  finetune={RUN_FINETUNE}  sequential={RUN_SEQUENTIAL}')
print(f'  Pretrain       : download={RUN_PRETRAIN_DOWNLOAD}  train={RUN_PRETRAIN_TRAIN}')
print(f'  Shared data    : {CMS_SHARED_DIR}')
print(f'  Run outputs    : {REPO_DIR}/{OUTPUT_ROOT}/cms/{{model_tag}}/')

## Cell 2 — Install dependencies

In [ ]:
!pip install -q vllm
!pip install -q peft accelerate datasets fire tqdm
!pip install -q "transformers>=4.40.0"
!pip install -q "torchao>=0.16.0"

# Install the SubPop library in editable mode
!cd {REPO_DIR}/subpop-main && pip install -q -e .

print('All dependencies installed.')

## Cell 3 — HuggingFace login

In Colab: click the **🔑 key icon** in the left sidebar → Add secret → Name: `HF_TOKEN` → Value: `hf_...`  
This keeps your token out of the notebook and off GitHub.

In [ ]:
from google.colab import userdata
from huggingface_hub import login

HF_TOKEN = userdata.get('HF_TOKEN')
login(HF_TOKEN)
print('HuggingFace login successful.')
print(f'Models to run: {ACTIVE_MODELS}')
for _m in ACTIVE_MODELS:
    print(f'  {_m}: {MODELS[_m]}')

## Cell 3b — (Optional) Download & filter SubPop pre-training data

Downloads `subpop_train.jsonl` (~33 MB) from HuggingFace and keeps 11 selected waves with thematic overlap to congestion pricing (climate, WFH/work culture, economic burden, gig economy).

**Before running:** accept the dataset terms at https://huggingface.co/datasets/jjssuh/subpop  
then re-run Cell 3 so your token is active.

This cell is **model-agnostic** — the filtered CSV is shared across both Llama and Qwen.

In [ ]:
_pretrain_csv = f'{REPO_DIR}/subpop-main/data/subpop-pretrain/processed/subpop-pretrain.csv'

if not RUN_PRETRAIN_DOWNLOAD:
    print('[SKIP] Pretrain download (3b): RUN_PRETRAIN_DOWNLOAD=False')
elif not should_run('Pretrain download (3b)', _pretrain_csv, FORCE['pretrain_download']):
    pass
else:
    from huggingface_hub import hf_hub_download
    import json, pandas as pd

    raw_path = hf_hub_download(
        repo_id='jjssuh/subpop',
        filename='subpop_train.jsonl',
        repo_type='dataset',
        local_dir=f'{REPO_DIR}/subpop-main/data/subpop-pretrain/raw',
    )
    print(f'Downloaded: {raw_path}')

    rows = []
    with open(raw_path) as f:
        for line in f:
            rows.append(json.loads(line))
    df = pd.DataFrame(rows)
    print(f'Full dataset: {len(df):,} rows | columns: {list(df.columns)}')

    # Pew Research waves selected for thematic overlap with congestion pricing opinion questions
    RELEVANT_WAVES = {67, 77, 81, 89, 94, 102, 103, 106, 108, 121, 128}

    def extract_wave(qkey):
        try:
            return int(str(qkey).split('_W')[-1])
        except Exception:
            return None

    df['wave'] = df['qkey'].apply(extract_wave)
    df_filtered = df[df['wave'].isin(RELEVANT_WAVES)].drop(columns=['wave'])

    print(f'\nFiltered to {len(RELEVANT_WAVES)} waves:')
    for w in sorted(RELEVANT_WAVES):
        subset = df[df['wave'] == w]
        print(f'  W{w}: {subset["qkey"].nunique()} questions, {len(subset)} rows')
    print(f'Filtered total: {len(df_filtered):,} rows | {df_filtered["qkey"].nunique()} unique questions')

    out_dir = f'{REPO_DIR}/subpop-main/data/subpop-pretrain/processed'
    os.makedirs(out_dir, exist_ok=True)
    df_filtered.to_csv(_pretrain_csv, index=False)
    print(f'Saved: {_pretrain_csv}')

## Cell 3c — (Optional) Per-model QA pre-training

Converts the filtered SubPop data to fine-tuning format, then runs QA pre-training for
each model in `ACTIVE_MODELS`. The resulting per-model checkpoint is used in Cell 7 as the
starting point for sequential CMS fine-tuning (`--from_peft_checkpoint`).

**Runtime:** ~15–30 min per model on L4 (3 epochs, 5k rows).

In [ ]:
import glob as _glob, sys

if not RUN_PRETRAIN_TRAIN:
    print('[SKIP] Pretrain training (3c): RUN_PRETRAIN_TRAIN=False')
else:
    # ── Shared data preparation (run once, model-agnostic) ───────────────────
    _pretrain_csv = f'{REPO_DIR}/subpop-main/data/subpop-pretrain/processed/subpop-pretrain.csv'
    require_exists(_pretrain_csv, 'Cell 3b (SubPop data download)')

    _qa_train = f'{REPO_DIR}/subpop-main/data/subpop-pretrain/processed/opnqa_QA_train.csv'
    if not exists_nonempty(_qa_train) or FORCE['pretrain_train']:
        print('Preparing fine-tuning format from filtered SubPop data...')
        import pandas as pd
        result_prep = subprocess.run(
            [
                sys.executable,
                'scripts/data_generation/prepare_finetuning_data.py',
                '--dataset', 'subpop-pretrain',
                '--steer_demographics_file_path', 'data/subpopulation_metadata/demographics_61.csv',
                '--train_ratio', '0.9',
                '--val_ratio', '0.1',
                '--test_ratio', '0.0',
                '--seed', '42',
            ],
            cwd=f'{REPO_DIR}/subpop-main',
            capture_output=False,
            text=True,
        )
        if result_prep.returncode != 0:
            raise RuntimeError('prepare_finetuning_data.py failed — see output above')

        # Subsample to keep Colab runtime manageable
        out_dir = f'{REPO_DIR}/subpop-main/data/subpop-pretrain/processed'
        for fmt in ['QA', 'BIO', 'PORTRAY', 'ALL']:
            train_path = f'{out_dir}/opnqa_{fmt}_train.csv'
            if os.path.exists(train_path):
                df = pd.read_csv(train_path)
                if len(df) > 5000:
                    df.sample(n=5000, random_state=42).to_csv(train_path, index=False)
                    print(f'  {fmt} train: subsampled -> 5,000 rows')
            val_path = f'{out_dir}/opnqa_{fmt}_val.csv'
            if os.path.exists(val_path):
                dv = pd.read_csv(val_path)
                if len(dv) > 300:
                    dv.sample(n=300, random_state=42).to_csv(val_path, index=False)
                    print(f'  {fmt} val:   subsampled -> 300 rows')
        print('Data preparation complete.')
    else:
        print(f'[SKIP]  Data preparation: {_qa_train} already exists.')

    # ── Per-model pre-training ───────────────────────────────────────────────
    for mtag in ACTIVE_MODELS:
        model_name = MODELS[mtag]
        pt_ckpt_dir  = f'{REPO_DIR}/subpop-main/data/subpop-pretrain/{mtag}/checkpoints_QA'
        pt_path_file = cms_local('pretrain_checkpoint_path.txt', mtag)

        if not should_run(f'Pretrain ({mtag})', [pt_path_file, pt_ckpt_dir], FORCE['pretrain_train']):
            continue

        print(f'\n[{mtag.upper()}] Starting QA pre-training with {model_name} ...')
        result_pt = subprocess.run(
            [
                sys.executable, 'scripts/experiment/run_finetune.py',
                '--model_name',              model_name,
                '--dataset',                 'subpop_pretrain_dataset',
                '--steering_type',           'QA',
                '--output_dir',              pt_ckpt_dir + '/',
                '--num_epochs',              '3',
                '--early_stopping_patience', '3',
                '--batch_size_training',     '1',
                '--lr',                      '2e-4',
                '--use_peft',
                '--loss_function_type',      'ce',
                '--enable_fsdp=False',
                '--use_wandb=False',
            ],
            cwd=f'{REPO_DIR}/subpop-main',
            env=make_cuda_env(),
            capture_output=False,
            text=True,
        )
        if result_pt.returncode != 0:
            raise RuntimeError(f'[{mtag}] Pre-training FAILED (exit {result_pt.returncode})')

        pt_dirs = sorted(_glob.glob(f'{pt_ckpt_dir}/*/'))
        if not pt_dirs:
            raise FileNotFoundError(f'[{mtag}] No pretrain checkpoint found after training.')
        pt_ckpt = pt_dirs[-1].rstrip('/')

        os.makedirs(cms_local('', mtag), exist_ok=True)
        with open(pt_path_file, 'w') as pf:
            pf.write(pt_ckpt)
        print(f'[{mtag}] Pretrain checkpoint: {pt_ckpt}')

        save_to_drive(
            pt_ckpt_dir,
            f'pretrain_checkpoints/{mtag}/checkpoints_QA',
            f'{mtag} pretrain QA checkpoint',
        )
        save_cms('pretrain_checkpoint_path.txt', f'{mtag} pretrain path', mtag)
        print(f'[{mtag}] Pre-training complete.')

## Cell 4 — Zero-shot baselines

Runs `step2_vllm_baselines.py` in zero-shot mode for each model in `ACTIVE_MODELS`.
Input metadata is read from `CMS_SHARED_DIR` (permanent shared dir).
Outputs go to each model's run directory: `cms/{model_tag}/distributions_{fmt}.csv`.

**Runtime:** ~10 min per model per format on L4 GPU.

In [ ]:
import sys

if not RUN_ZEROSHOT:
    print('[SKIP] Zero-shot baselines: RUN_ZEROSHOT=False')
else:
    for mtag in ACTIVE_MODELS:
        model_name = MODELS[mtag]

        # Find which formats still need to be run for this model
        if FORCE['zeroshot']:
            run_fmts = ACTIVE_FORMATS
        else:
            run_fmts = [
                f for f in ACTIVE_FORMATS
                if not exists_nonempty(cms_local(f'distributions_{f}.csv', mtag))
            ]

        if not run_fmts:
            print(f'[SKIP]  Zero-shot {mtag}: all formats present.')
            continue

        print(f'\n[RUN]  Zero-shot {mtag} ({model_name}) -> formats: {run_fmts}')
        os.makedirs(cms_local('', mtag), exist_ok=True)

        # Input metadata from shared permanent dir; output to model-specific dir
        result = subprocess.run(
            [
                sys.executable, 'step2_vllm_baselines.py',
                '--mode',             'zeroshot',
                '--model_name',       model_name,
                '--demographics_csv', f'{CMS_SHARED_REL}/cms_demographics.csv',
                '--steering_json',    f'{CMS_SHARED_REL}/cms_steering_prompts.json',
                '--questions_json',   f'{CMS_SHARED_REL}/cms_questions.json',
                '--output_dir',       cms_rel('', mtag),
                '--formats',          *run_fmts,
            ],
            cwd=REPO_DIR,
            env=make_cuda_env(),
            capture_output=False,
            text=True,
        )
        if result.returncode != 0:
            raise RuntimeError(f'[{mtag}] Zero-shot baselines FAILED (exit {result.returncode})')

        print(f'Saving {mtag} zero-shot distributions to Drive...')
        for fmt in run_fmts:
            rel = f'distributions_{fmt}.csv'
            if os.path.exists(cms_local(rel, mtag)):
                save_cms(rel, rel, mtag)

        print(f'[{mtag}] Zero-shot complete.')

## Cell 4b — Statistical bounds (shared, model-agnostic)

Computes uniform upper bound and bootstrap lower bound using WD for ordinal questions and TVD for nominal questions.
Written to `CMS_SHARED_DIR/statistical_bounds.csv` (shared, not per-model).
The evaluation cell (Cell 8) copies it into each model's output directory before running step6,
so step6's auto-discovery logic finds it correctly.

In [ ]:
import sys

# Bounds are model-agnostic; write to shared permanent dir
_bounds = f'{CMS_SHARED_DIR}/statistical_bounds.csv'

if not should_run('Statistical bounds', _bounds, FORCE['bounds']):
    pass
else:
    result = subprocess.run(
        [
            sys.executable, 'step2_vllm_baselines.py',
            '--mode',             'bounds',
            '--ground_truth_csv', f'{CMS_SHARED_REL}/cms_survey_distributions.csv',
            '--questions_json',   f'{CMS_SHARED_REL}/cms_questions.json',
            '--output_dir',       CMS_SHARED_REL,
        ],
        cwd=REPO_DIR,
        capture_output=False,
        text=True,
    )
    if result.returncode != 0:
        raise RuntimeError(f'Statistical bounds FAILED (exit {result.returncode})')

    if os.path.exists(_bounds):
        print(f'Statistical bounds saved to {_bounds}')
    print('Statistical bounds complete.')

## Cell 5 — Step 3: Prepare CMS fine-tuning data (shared, model-agnostic)

Converts CMS ground-truth distributions into training rows for each active format.
Output goes to `approach2_outputs/cms/finetuning_data/` — the **permanent undated path**
that `cms_dataset` in `subpop-main/subpop/train/configs/datasets.py` is hardcoded to read.

Both Llama and Qwen fine-tuning read the same CSVs from this shared location.

Produces: `cms_{fmt}_train.csv`, `cms_{fmt}_val.csv`, `cms_{fmt}_test.csv` for each active format, plus `ALL`.

In [ ]:
import sys

# All inputs and outputs use CMS_SHARED_REL (permanent undated path)
# This is required because cms_dataset in datasets.py hardcodes:
#   ../approach2_outputs/cms/finetuning_data/cms_{steering_type}_train.csv
_s3_formats = list(dict.fromkeys([*ACTIVE_FORMATS, 'ALL']))
_s3_outputs = [
    f'{CMS_SHARED_DIR}/finetuning_data/cms_{_fmt}_{_split}.csv'
    for _fmt in _s3_formats
    for _split in ['train', 'val', 'test']
]

if not should_run('Step 3: fine-tuning data prep', _s3_outputs, FORCE['step3']):
    pass
else:
    os.makedirs(f'{CMS_SHARED_DIR}/finetuning_data', exist_ok=True)

    result = subprocess.run(
        [
            sys.executable, 'step3_prepare_finetuning_data.py',
            '--distributions_csv',   f'{CMS_SHARED_REL}/cms_survey_distributions.csv',
            '--demographics_csv',    f'{CMS_SHARED_REL}/cms_demographics.csv',
            '--steering_json',       f'{CMS_SHARED_REL}/cms_steering_prompts.json',
            '--question_split_json', f'{CMS_SHARED_REL}/cms_question_split.json',
            '--output_dir',          f'{CMS_SHARED_REL}/finetuning_data',
            '--formats',             *_s3_formats,
        ],
        cwd=REPO_DIR,
        capture_output=False,
        text=True,
    )
    if result.returncode != 0:
        raise RuntimeError(f'Step 3 fine-tuning data prep FAILED (exit {result.returncode})')

    print('Fine-tuning data written to:')
    for _f in sorted(os.listdir(f'{CMS_SHARED_DIR}/finetuning_data')):
        print(f'  {_f}')
    print('Step 3 complete.')

## Cell 6 — Base CMS fine-tuning + inference

Runs LoRA fine-tuning on CMS data for each `(model, format)` pair, then runs inference
on the held-out test set. Both models read training data from the shared `finetuning_data/`.

Outputs per `(model, format)` pair:
- Checkpoint: `cms/{model_tag}/checkpoints_{fmt}/`
- Predictions: `cms/{model_tag}/results_finetuned_{fmt}.csv`

**Runtime:** ~30 min per (model, format) on L4 GPU.

In [ ]:
import pathlib, sys

if not RUN_FINETUNE:
    print('[SKIP] Base fine-tuning: RUN_FINETUNE=False')
else:
    for mtag in ACTIVE_MODELS:
        model_name = MODELS[mtag]
        print(f'\n{"="*60}')
        print(f'BASE FINE-TUNE: {mtag.upper()} ({model_name})')
        print(f'{"="*60}')

        for fmt in ACTIVE_FORMATS:
            ckpt_dir  = cms_local(f'checkpoints_{fmt}', mtag)
            infer_out = cms_local(f'results_finetuned_{fmt}.csv', mtag)
            lora_name = f'cms_{fmt}'
            # Test CSV is read from shared permanent dir (set by step3, matches cms_dataset)
            test_csv_local = f'{CMS_SHARED_DIR}/finetuning_data/cms_{fmt}_test.csv'
            test_csv_rel   = f'{CMS_SHARED_REL}/finetuning_data/cms_{fmt}_test.csv'

            # ── Fine-tuning ──────────────────────────────────────────────────
            if not should_run(f'{mtag} {fmt} fine-tune', ckpt_dir, FORCE['finetune']):
                pass
            else:
                os.makedirs(cms_local('', mtag), exist_ok=True)
                result_ft = subprocess.run(
                    [
                        sys.executable, 'scripts/experiment/run_finetune.py',
                        '--model_name',              model_name,
                        '--dataset',                 'cms_dataset',
                        '--steering_type',           fmt,
                        '--output_dir',              f'../{cms_rel(f"checkpoints_{fmt}", mtag)}/',
                        '--num_epochs',              '20',
                        '--early_stopping_patience', '3',
                        '--batch_size_training',     '1',
                        '--lr',                      '2e-4',
                        '--use_peft',
                        '--loss_function_type',      'ce',
                        '--enable_fsdp=False',
                        '--use_wandb=False',
                    ],
                    cwd=f'{REPO_DIR}/subpop-main',
                    env=make_cuda_env(),
                    capture_output=False,
                    text=True,
                )
                if result_ft.returncode != 0:
                    raise RuntimeError(f'[{mtag} {fmt}] Fine-tuning FAILED (exit {result_ft.returncode})')
                print(f'[{mtag} {fmt}] Fine-tuning complete.')
                save_to_drive(ckpt_dir, cms_drive_rel(f'checkpoints_{fmt}', mtag), f'{mtag} {fmt} checkpoint')

            # ── Inference ────────────────────────────────────────────────────
            if not should_run(f'{mtag} {fmt} inference', infer_out, FORCE['finetune']):
                continue

            require_exists(test_csv_local, 'Cell 5 (step3 finetuning data prep)')
            ckpt_path = latest_subdir(cms_local(f'checkpoints_{fmt}/*', mtag))
            require_exists(ckpt_path, f'Cell 6 ({mtag} {fmt} fine-tuning)')
            ckpt_rel = os.path.relpath(ckpt_path, f'{REPO_DIR}/subpop-main')
            print(f'[{mtag} {fmt}] Running inference from {ckpt_path}')

            result_infer = subprocess.run(
                [
                    sys.executable, 'scripts/experiment/run_inference.py',
                    '--input_paths',             f'../{test_csv_rel}',
                    '--output_dir',              f'../{cms_rel("", mtag)}/',
                    '--base_model_name_or_path', model_name,
                    '--lora_path',               ckpt_rel,
                    '--lora_name',               lora_name,
                ],
                cwd=f'{REPO_DIR}/subpop-main',
                env=make_cuda_env(),
                capture_output=False,
                text=True,
            )
            if result_infer.returncode != 0:
                raise RuntimeError(f'[{mtag} {fmt}] Inference FAILED (exit {result_infer.returncode})')

            raw_stem = pathlib.Path(test_csv_rel).stem          # e.g. 'cms_QA_test'
            raw_out  = cms_local(f'{raw_stem}_{lora_name}.csv', mtag)
            canonical = cms_local(f'results_finetuned_{fmt}.csv', mtag)
            if os.path.exists(raw_out):
                os.rename(raw_out, canonical)
                print(f'[{mtag} {fmt}] Renamed -> results_finetuned_{fmt}.csv')
            else:
                candidates = [f for f in os.listdir(cms_local('', mtag)) if f'cms_{fmt}_test' in f]
                raise FileNotFoundError(
                    f'[{mtag} {fmt}] Expected inference output not found: {raw_out}\n'
                    f'Candidates: {candidates}'
                )

            save_cms(f'results_finetuned_{fmt}.csv', f'{mtag} results_finetuned_{fmt}.csv', mtag)

    print('\nBase fine-tuning + inference complete for all active (model, format) pairs.')

## Cell 7 — Sequential fine-tuning + inference

Starts from the per-model QA pre-train checkpoint (Cell 3c) and continues fine-tuning on
each CMS format, producing a sequential variant for each `(model, format)` pair.

**Prerequisite:** Cell 3c must have run and produced `cms/{model_tag}/pretrain_checkpoint_path.txt`.

Outputs per `(model, format)` pair:
- Checkpoint: `cms/{model_tag}/checkpoints_{fmt}_seq/`
- Predictions: `cms/{model_tag}/results_finetuned_{fmt}_seq.csv`

**Runtime:** ~30 min per (model, format) on L4 GPU.

In [ ]:
import pathlib, sys

if not RUN_SEQUENTIAL:
    print('[SKIP] Sequential fine-tuning: RUN_SEQUENTIAL=False')
else:
    for mtag in ACTIVE_MODELS:
        model_name   = MODELS[mtag]
        pt_path_file = cms_local('pretrain_checkpoint_path.txt', mtag)

        try:
            require_exists(pt_path_file, 'Cell 3c (per-model QA pre-training)')
        except FileNotFoundError as e:
            print(f'[SKIP] Sequential {mtag}: {e}')
            continue

        with open(pt_path_file) as pf:
            pretrain_ckpt = pf.read().strip()

        try:
            require_exists(pretrain_ckpt, 'Cell 3c (pretrain checkpoint dir)')
        except FileNotFoundError as e:
            print(f'[SKIP] Sequential {mtag}: checkpoint dir missing — {e}')
            continue

        print(f'\n{"="*60}')
        print(f'SEQUENTIAL FINE-TUNE: {mtag.upper()} ({model_name})')
        print(f'  Pretrain checkpoint: {pretrain_ckpt}')
        print(f'{"="*60}')

        for fmt in ACTIVE_FORMATS:
            seq_ckpt_dir  = cms_local(f'checkpoints_{fmt}_seq', mtag)
            infer_out     = cms_local(f'results_finetuned_{fmt}_seq.csv', mtag)
            lora_name     = f'cms_{fmt}_seq'
            test_csv_local = f'{CMS_SHARED_DIR}/finetuning_data/cms_{fmt}_test.csv'
            test_csv_rel   = f'{CMS_SHARED_REL}/finetuning_data/cms_{fmt}_test.csv'

            # ── Sequential fine-tuning ───────────────────────────────────────
            if not should_run(f'{mtag} {fmt} seq fine-tune', seq_ckpt_dir, FORCE['sequential']):
                pass
            else:
                os.makedirs(cms_local('', mtag), exist_ok=True)
                result_ft = subprocess.run(
                    [
                        sys.executable, 'scripts/experiment/run_finetune.py',
                        '--model_name',              model_name,
                        '--dataset',                 'cms_dataset',
                        '--steering_type',           fmt,
                        '--output_dir',              f'../{cms_rel(f"checkpoints_{fmt}_seq", mtag)}/',
                        '--num_epochs',              '20',
                        '--early_stopping_patience', '3',
                        '--batch_size_training',     '1',
                        '--lr',                      '2e-4',
                        '--use_peft',
                        '--loss_function_type',      'ce',
                        '--enable_fsdp=False',
                        '--use_wandb=False',
                        '--from_peft_checkpoint',    pretrain_ckpt,
                    ],
                    cwd=f'{REPO_DIR}/subpop-main',
                    env=make_cuda_env(),
                    capture_output=False,
                    text=True,
                )
                if result_ft.returncode != 0:
                    raise RuntimeError(f'[{mtag} {fmt}_seq] Fine-tuning FAILED (exit {result_ft.returncode})')
                print(f'[{mtag} {fmt}_seq] Fine-tuning complete.')
                save_to_drive(seq_ckpt_dir, cms_drive_rel(f'checkpoints_{fmt}_seq', mtag), f'{mtag} {fmt}_seq checkpoint')

            # ── Sequential inference ─────────────────────────────────────────
            if not should_run(f'{mtag} {fmt} seq inference', infer_out, FORCE['sequential']):
                continue

            require_exists(test_csv_local, 'Cell 5 (step3 finetuning data prep)')
            seq_ckpt = latest_subdir(cms_local(f'checkpoints_{fmt}_seq/*', mtag))
            require_exists(seq_ckpt, f'Cell 7 ({mtag} {fmt}_seq fine-tuning)')
            seq_ckpt_rel = os.path.relpath(seq_ckpt, f'{REPO_DIR}/subpop-main')
            print(f'[{mtag} {fmt}_seq] Running inference from {seq_ckpt}')

            result_infer = subprocess.run(
                [
                    sys.executable, 'scripts/experiment/run_inference.py',
                    '--input_paths',             f'../{test_csv_rel}',
                    '--output_dir',              f'../{cms_rel("", mtag)}/',
                    '--base_model_name_or_path', model_name,
                    '--lora_path',               seq_ckpt_rel,
                    '--lora_name',               lora_name,
                ],
                cwd=f'{REPO_DIR}/subpop-main',
                env=make_cuda_env(),
                capture_output=False,
                text=True,
            )
            if result_infer.returncode != 0:
                raise RuntimeError(f'[{mtag} {fmt}_seq] Inference FAILED (exit {result_infer.returncode})')

            raw_stem  = pathlib.Path(test_csv_rel).stem
            raw_out   = cms_local(f'{raw_stem}_{lora_name}.csv', mtag)
            canonical = cms_local(f'results_finetuned_{fmt}_seq.csv', mtag)
            if os.path.exists(raw_out):
                os.rename(raw_out, canonical)
                print(f'[{mtag} {fmt}_seq] Renamed -> results_finetuned_{fmt}_seq.csv')
            else:
                candidates = [f for f in os.listdir(cms_local('', mtag)) if f'cms_{fmt}_test' in f]
                raise FileNotFoundError(
                    f'[{mtag} {fmt}_seq] Expected inference output not found: {raw_out}\n'
                    f'Candidates: {candidates}'
                )

            save_cms(f'results_finetuned_{fmt}_seq.csv', f'{mtag} results_finetuned_{fmt}_seq.csv', mtag)

    print('\nSequential fine-tuning + inference complete for all active (model, format) pairs.')

## Cell 8 — Evaluation (per model)

Runs `step6_full_evaluation.py` once per active model.

**Key setup before step6 runs:**
- `statistical_bounds.csv` is copied from `CMS_SHARED_DIR` into each model's output directory
  so step6's auto-discovery (which looks for bounds in `predictions_dir`) finds it.
- All shared input CSVs (ground truth, questions, split) are referenced from `CMS_SHARED_DIR`.

**Output per model:** `cms/{model_tag}/evaluation/`
- `ablation_table.csv` — distance comparison across all discovered methods
- `descriptive_all_rows/` — per-row distance outputs for all questions
- `test_only_fair/` — fair comparison restricted to test questions only

In [ ]:
import sys

# Shared bounds file (model-agnostic, written by Cell 4b)
_shared_bounds = f'{CMS_SHARED_DIR}/statistical_bounds.csv'

for mtag in ACTIVE_MODELS:
    _eval_outputs = [
        cms_local('evaluation/ablation_table.csv', mtag),
        cms_local('evaluation/method_coverage.csv', mtag),
        cms_local('evaluation/test_only_fair', mtag),
    ]
    if not should_run(f'Evaluation ({mtag})', _eval_outputs, FORCE['eval']):
        continue

    print(f'\n[RUN]  Evaluation: {mtag.upper()}')
    os.makedirs(cms_local('evaluation', mtag), exist_ok=True)

    # Copy statistical_bounds.csv into model's output dir so step6 discovers it.
    # step6 looks for: predictions_dir / 'statistical_bounds.csv'
    if os.path.exists(_shared_bounds):
        shutil.copy2(_shared_bounds, cms_local('statistical_bounds.csv', mtag))
        print(f'  Copied statistical_bounds.csv -> cms/{mtag}/')
    else:
        print('  WARNING: statistical_bounds.csv not found (run Cell 4b first). Bounds will be omitted.')

    result = subprocess.run(
        [
            sys.executable, 'step6_full_evaluation.py',
            '--ground_truth_csv',    f'{CMS_SHARED_REL}/cms_survey_distributions.csv',
            '--questions_json',      f'{CMS_SHARED_REL}/cms_questions.json',
            '--question_split_json', f'{CMS_SHARED_REL}/cms_question_split.json',
            '--predictions_dir',     cms_rel('', mtag),
            '--weights_csv',         f'{CMS_SHARED_REL}/cms_subgroup_weights.csv',
            '--output_dir',          cms_rel('evaluation', mtag),
        ],
        cwd=REPO_DIR,
        capture_output=False,
        text=True,
    )
    if result.returncode != 0:
        raise RuntimeError(f'[{mtag}] Evaluation FAILED (exit {result.returncode})')

    print(f'\n[{mtag}] Evaluation outputs:')
    for _f in sorted(os.listdir(cms_local('evaluation', mtag))):
        print(f'  {_f}')

    print(f'Saving {mtag} evaluation to Drive...')
    shutil.copytree(
        cms_local('evaluation', mtag),
        f'{DRIVE_BASE}/{cms_drive_rel("evaluation", mtag)}',
        dirs_exist_ok=True,
    )
    print(f'  checkmark {mtag} evaluation/')

    # Back up per-model result CSVs
    for _f in os.listdir(cms_local('', mtag)):
        if _f.startswith('results_') and _f.endswith('.csv'):
            save_cms(_f, _f, mtag)

print('\nAll evaluations complete.')